# ML & DL training

Single notebook for **FLAML AutoML** (feature CSVs) and **DL training** (Parquet + splits).

- **Data**: `eyefeatures.data` (Parquet + meta); split info from `benchmark_experiments` (e.g. `features_output/splits/`).
- **Training logic**: `utils/flaml_training.py`, `utils/dl_training_utils.py`.

Run **create_splits** and **feature_extraction_all** first so splits and feature files exist.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from utils.flaml_training import run_training_battery as run_flaml_battery, print_results_summary as print_flaml_summary
from utils.dl_training_utils import (
    run_dl_training_battery,
    print_results_summary as print_dl_summary,
    ALL_REPRESENTATIONS,
    DEFAULT_TIMESERIES_FEATURES,
    DEFAULT_MAX_LENGTH,
    DEFAULT_IMAGE_SHAPE,
)

from utils.benchmark_utils import get_benchmark_dir, find_datasets_parquet, load_dataset_parquet
from utils.feature_extraction_utils import setup_paths

In [2]:
# Shared paths
paths = setup_paths(output_dir='features_output', splits_dir='features_output/splits')
BENCHMARK_DIR = paths['benchmark_dir']
FEATURES_DIR = paths['output_dir']
SPLITS_DIR = paths['splits_dir']

Path('flaml_results').mkdir(exist_ok=True)
Path('dl_results').mkdir(exist_ok=True)

print(f"Benchmark: {BENCHMARK_DIR}")
print(f"Features: {FEATURES_DIR}")
print(f"Splits: {SPLITS_DIR}")

Benchmark: C:\Users\LEGION\EyeFeatures\data\benchmark
Features: features_output
Splits: features_output\splits


## Part 1: ML (FLAML AutoML)

Trains on precomputed feature train/val/test CSVs from `features_output/`.

In [ ]:
FLAML_RESULTS_FILE = Path('results') / 'flaml_results_all_batteries.csv'
TIME_BUDGET = 300
FEATURE_BATTERIES = ['simple_features', 'extended_features', 'complex_features', 'distance_features']

results_flaml = run_flaml_battery(
    FEATURES_DIR,
    SPLITS_DIR,
    FLAML_RESULTS_FILE,
    FEATURE_BATTERIES,
    time_budget=TIME_BUDGET,
)

print_flaml_summary(results_flaml)

## Part 2: DL training

Uses Parquet datasets and split info; fit on 2d, timeseries, merged representations.

In [ ]:
DL_RESULTS_FILE = Path('results') / 'dl_training_results.csv'
load_func = lambda p: load_dataset_parquet(p, benchmark_dir=BENCHMARK_DIR)

results_dl = run_dl_training_battery(
    BENCHMARK_DIR,
    SPLITS_DIR,
    DL_RESULTS_FILE,
    find_datasets_parquet,
    load_func,
    dataset_types=['2d', 'timeseries', 'merged'],
    representations=ALL_REPRESENTATIONS,
    cnn_architecture='large_resnet',
    max_epochs=30,
    batch_size=32,
    image_shape=DEFAULT_IMAGE_SHAPE,
    timeseries_features=DEFAULT_TIMESERIES_FEATURES,
    max_length=DEFAULT_MAX_LENGTH,
    skip_existing=True,
)

print_dl_summary(results_dl)

Found 47 datasets to process


Datasets:   0%|          | 0/47 [00:00<?, ?it/s]


Dataset: ASD_fixations (63,627 rows, 1 split(s))

  Label: ASD_label
    Training 2d/heatmap_fixed... Number of channels = 1.


100%|██████████| 7/7 [00:00<00:00, 247.63it/s]
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

   | Name            | Type             | Params | Mode 
--------------------------------------------------------------
0  | backbone        | Sequential       | 2.4 M  | train
1  | head            | ModuleList       | 0      | train
2  | loss_fn         | CrossEntropyLoss | 0      | train
3  | flat            | Flatten          | 0      | train
4  | accuracy        | BinaryAccuracy   | 0      | train
5  | precision       | BinaryPrecision  | 0      | train
6  | recall          | BinaryRecall     | 0      | train
7  | f1              | BinaryF1Score    | 0      | train
8  | macro_precision | BinaryPrecision  | 0      | train
9  | macro_recall    | BinaryRecall     | 0      | train
10 | macro_f1        | BinaryF1Score    | 0      | train
11 | prob            | Softmax          | 0      | train
-------------------------------------